# Uncertainty under distribution shift

This notebook studies two controlled stress tests: decreasing spatial support through buffered LOBO and synthetic corruption through feature permutation.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import bovino_results as br

## Spatial exclusion buffer

In [ ]:
buffer_metrics = br.load_buffer_metrics()
selected = buffer_metrics[buffer_metrics['method'].isin(
    ['mc_dropout', 'deep_ensemble', 'subsample_ensemble', 'llla', 'tabicl_hidden']
)]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for method, group in selected.groupby('method'):
    group = group.sort_values('buffer_m')
    axes[0].plot(group.buffer_m, group.accuracy, marker='o', label=method)
    axes[1].plot(group.buffer_m, group.error_auroc_total_uncertainty, marker='o', label=method)
axes[0].set(title='Accuracy', xlabel='Exclusion buffer (m)', ylabel='Score')
axes[1].set(title='Error AUROC: predictive entropy', xlabel='Exclusion buffer (m)', ylabel='Score')
for ax in axes:
    ax.grid(alpha=.25)
axes[1].legend(fontsize=8)
plt.show()

## Feature perturbation

In [ ]:
perturbations = br.load_perturbation_metrics()
summary = perturbations.groupby(['method', 'control'])[
    ['accuracy', 'mean_total_uncertainty', 'mean_epistemic_uncertainty']
].mean().round(3)
display(summary)

## Interpretation

The buffer experiment approximates loss of local spatial support. Feature permutation is a synthetic diagnostic, not a realistic deployment distribution. A useful response combines declining accuracy with increasing uncertainty, but the uncertainty ranking must also be checked through Error AUROC or Error AUPRC.